In [1]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [2]:
# ==================================================
# Import Libraries
# ==================================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from src.mlflow.tracking import MLflowTracker
from src.mlflow.utils import classification_metrics

d:\Subject\CV2026\Market Risk Classification\market-risk-classification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==================================================
# Load Data
# ==================================================

df = pd.read_csv(
    r"D:\Subject\CV2026\Market Risk Classification\market-risk-classification\data\processed\BTCUSDT\BTCUSDT_1m_target.csv"
)

df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore,target
0,2025-01-01 00:00:00,93576.00,93610.93,93537.50,93610.93,8.21827,2025-01-01 00:00:59.999,7.689788e+05,2631,3.95157,369757.326529,0,1
1,2025-01-01 00:01:00,93610.93,93652.00,93606.20,93652.00,12.14029,2025-01-01 00:01:59.999,1.136551e+06,1273,4.08887,382791.500172,0,1
2,2025-01-01 00:02:00,93652.00,93702.15,93635.98,93702.15,11.60597,2025-01-01 00:02:59.999,1.087101e+06,1095,5.86840,549682.868570,0,0
3,2025-01-01 00:03:00,93702.14,93702.15,93654.48,93677.98,8.72958,2025-01-01 00:03:59.999,8.177203e+05,1461,2.48203,232486.113080,0,0
4,2025-01-01 00:04:00,93677.98,93677.99,93659.92,93661.20,5.24749,2025-01-01 00:04:59.999,4.915570e+05,988,0.48880,45786.251963,0,1


In [4]:
# ==================================================
# Data Information
# ==================================================

print(df.shape)

df.info()

(20160, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20160 entries, 0 to 20159
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   open_time               20160 non-null  object 
 1   open                    20160 non-null  float64
 2   high                    20160 non-null  float64
 3   low                     20160 non-null  float64
 4   close                   20160 non-null  float64
 5   volume                  20160 non-null  float64
 6   close_time              20160 non-null  object 
 7   quote_asset_volume      20160 non-null  float64
 8   number_of_trades        20160 non-null  int64  
 9   taker_buy_base_volume   20160 non-null  float64
 10  taker_buy_quote_volume  20160 non-null  float64
 11  ignore                  20160 non-null  int64  
 12  target                  20160 non-null  int64  
dtypes: float64(8), int64(3), object(2)
memory usage: 2.0+ MB


In [5]:
# ==================================================
# Drop Unnecessary Columns
# ==================================================

df = df.drop(
    columns=[
        "open_time",
        "close_time",
        "ignore"
    ]
)

df.head()

,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,target
0,93576.00,93610.93,93537.50,93610.93,8.21827,7.689788e+05,2631,3.95157,369757.326529,1
1,93610.93,93652.00,93606.20,93652.00,12.14029,1.136551e+06,1273,4.08887,382791.500172,1
2,93652.00,93702.15,93635.98,93702.15,11.60597,1.087101e+06,1095,5.86840,549682.868570,0
3,93702.14,93702.15,93654.48,93677.98,8.72958,8.177203e+05,1461,2.48203,232486.113080,0
4,93677.98,93677.99,93659.92,93661.20,5.24749,4.915570e+05,988,0.48880,45786.251963,1


In [6]:
# ==================================================
# Missing Values
# ==================================================

df.isnull().sum()

open                      0
high                      0
low                       0
close                     0
volume                    0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
target                    0
dtype: int64

In [7]:
# ==================================================
# Feature / Target
# ==================================================

TARGET = "target"


FEATURES = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
]


X = df[FEATURES]

y = df[TARGET]


print(X.shape)
print(y.shape)

(20160, 9)
(20160,)


In [8]:
# ==================================================
# Target Distribution
# ==================================================

y.value_counts()

target
0    10464
1     9696
Name: count, dtype: int64

In [9]:
# ==================================================
# Train Test Split
# ==================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,
    shuffle=False
)

print(X_train.shape)
print(X_test.shape)

(16128, 9)
(4032, 9)


In [10]:
# ==================================================
# Feature Scaling
# ==================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)


X_test_scaled = scaler.transform(
    X_test
)

In [11]:
# ==================================================
# Initialize MLflow Tracker
# ==================================================

from src.mlflow.tracking import MLflowTracker

tracker = MLflowTracker()

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(
    random_state=42,
    max_iter=1000,
)


with tracker.start_run(
    run_name="Logistic Regression Baseline"
):

    model.fit(
        X_train_scaled,
        y_train,
    )

    y_pred = model.predict(
        X_test_scaled,
    )

    y_prob = model.predict_proba(
        X_test_scaled,
    )[:, 1]


    metrics = classification_metrics(
        y_test,
        y_pred,
        y_prob,
    )


    # ============================
    # Log MLflow
    # ============================

    tracker.log_params(
        {
            "model": "LogisticRegression",
            "max_iter": 1000,
            "random_state": 42,
        }
    )


    tracker.log_metrics(
        metrics
    )


    tracker.log_model(
        model
    )
    tracker.generate_summary()

MLflow training summary generated
Saved at: D:\Subject\CV2026\Market Risk Classification\market-risk-classification\artifacts\training_summary.txt


In [13]:
# ==================================================
# Prediction
# ==================================================

y_pred = model.predict(
    X_test_scaled
)


y_prob = model.predict_proba(
    X_test_scaled
)[:,1]

In [14]:
# ==================================================
# Evaluation
# ==================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


metrics = {

    "accuracy": accuracy_score(
        y_test,
        y_pred
    ),

    "precision": precision_score(
        y_test,
        y_pred
    ),

    "recall": recall_score(
        y_test,
        y_pred
    ),

    "f1": f1_score(
        y_test,
        y_pred
    ),

    "roc_auc": roc_auc_score(
        y_test,
        y_prob
    )
}


metrics

{'accuracy': 0.5225694444444444,
 'precision': 0.5113895216400911,
 'recall': 0.2308483290488432,
 'f1': 0.31810131066241587,
 'roc_auc': 0.5197334952694056}

In [15]:
# ============================
# Calculate Metrics
# # ============================

metrics = classification_metrics(
    y_test,
    y_pred,
    y_prob
)

In [16]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "Market Risk Classification"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print(runs[[
    "run_id",
    "tags.mlflow.runName",
]])

                             run_id           tags.mlflow.runName
0  ff6af802a7f44681b44412c7f043a5ad  Logistic Regression Baseline
